In [ ]:
import polars as pl
import polars.selectors as cs
import re
from jsna_climate_and_environment.geography.places import SURREY_DISTRICTS
from jsna_climate_and_environment.datasets import iod25
import plotly.express as px
import fsspec

# Emmissions Data
commonly emmissions and climate change are related to climate reductions.

However the relationship with emmissions and inequality are interesting to consider
https://www.london.gov.uk/sites/default/files/2023-06/Air%20quality%20exposure%20and%20inequalities%20study%20-%20part%20two%20-%20Comparison%20with%20other%20cities.pdf

The above article suggests the potential benefit of comparing consumption and exposure to highlight inequalities and injustice at a population health 

- carbon footprint data sounds interesting https://www.carbon.place/data/
- 



In [ ]:
# https://naei.energysecurity.gov.uk
# the dashboard provides modelled estimates in a map view. this "heatmap" info can identify hotspots
# the underlying data has a few components.
# - 1km grid resolution numeric values in a geotiff or ASCII GRID format (rioxarray can be used to clip geotiff image into surrey polygons which is likely the most efficient approach)
# - point of source emmissions in a csv or shapefile (this can be used to identify poluting industrys)
# - different dataset every year (resulting in multiple downloads)
# - different datasets for every pollutant  (resulting in multiple downloads and difficulty interpreting or comparing risks)

# data from The Environment Agency's Pollution Inventory (PI) for England might be useful at a higher level
import xarray
from jsna_climate_and_environment.geography.places import read_gdb
import rioxarray
import fsspec
import geopandas as gpd
from geocube.api.core import make_geocube

FIRST_YEAR = 2005
LATEST_YEAR = 2023

POLUTION = "pm2_5"
NITROGEN_OXIDES = "nox"
SULPHUR_DIOXIDE = "so2"

lsoa_geom = read_gdb(layer="lsoa")
lsoa_geom["key"] = lsoa_geom["lsoa21_code"].str.replace('[^0-9]', '', regex=True).astype('int64')

def read_polution_inventory(polutant: str, year: int):
    version = "23"
    url = f"https://naei.energysecurity.gov.uk/data/maps/download-gridded-emissions/{version}/{polutant}/{year}"
    fs = fsspec.filesystem("zip", fo=f"simplecache::{url}")
    with fs.open(f"GeoTIFF_layers/total{polutant}{version}_{year}.tif") as file:
        raster = rioxarray.open_rasterio(file)
        assert not isinstance(raster, list)

        
pm_gdf = read_polution_inventory(POLUTION, 2023)
pm_gdf

<xarray.DataArray (band: 1, y: 45, x: 64)> Size: 12kB
array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]]], shape=(1, 45, 64), dtype=float32)
Coordinates:
  * band         (band) int64 8B 1
  * y            (y) float64 360B 1.755e+05 1.745e+05 ... 1.325e+05 1.315e+05
  * x            (x) float64 512B 4.805e+05 4.815e+05 ... 5.425e+05 5.435e+05
    spatial_ref  int64 8B 0
Attributes:
    DataType:            Generic
    AREA_OR_POINT:       Area
    RepresentationType:  ATHEMATIC
    scale_factor:        1.0
    add_offset:          0.0
    _FillValue:          0.0
<xarray.Dataset> Size: 7kB
Dimensions:      (y: 45, x: 64, key: 719)
Coordinates:
  * y            (y) float64 360B 1.755e+05 1.745e+05 ... 1.325e+05 1.315e+05
  * x            (x) float64 512B 4.805e+05 4.815e+05 ... 5.425e+05 5.435e

C:\Users\Nmolkent\AppData\Local\Temp\11\ipykernel_14080\4004501961.py:46: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  return lsoa_geom.merge(grouped_polution, on="key", copy=True)


,lsoa21_code,source_url,geometry,key,spatial_ref
0,E01030296,https://services1.arcgis.com/ESMARspQHYMw9BZ9/...,"MULTIPOLYGON (((517080.666 164767.597, 517080....",1030296,0
1,E01030296,https://services1.arcgis.com/ESMARspQHYMw9BZ9/...,"MULTIPOLYGON (((517080.666 164767.597, 517080....",1030296,0
2,E01030296,https://services1.arcgis.com/ESMARspQHYMw9BZ9/...,"MULTIPOLYGON (((517080.666 164767.597, 517080....",1030296,0
3,E01030296,https://services1.arcgis.com/ESMARspQHYMw9BZ9/...,"MULTIPOLYGON (((517080.666 164767.597, 517080....",1030296,0
4,E01030296,https://services1.arcgis.com/ESMARspQHYMw9BZ9/...,"MULTIPOLYGON (((517080.666 164767.597, 517080....",1030296,0
...,...,...,...,...,...
2076475,E01035364,https://services1.arcgis.com/ESMARspQHYMw9BZ9/...,"MULTIPOLYGON (((534472 158635.821, 534465.159 ...",1035364,0
2076476,E01035364,https://services1.arcgis.com/ESMARspQHYMw9BZ9/...,"MULTIPOLYGON (((534472 158635.821, 534465.159 ...",1035364,0
2076477,E01035364,https://services1.arcgis.com/ESMARspQHYMw9BZ9/...,"MULTIPOLYGON (((534472 158635.821, 534465.159 ...",1035364,0
2076478,E01035364,https://services1.arcgis.com/ESMARspQHYMw9BZ9/...,"MULTIPOLYGON (((534472 158635.821, 534465.159 ...",1035364,0


In [43]:
pm_gdf

,lsoa21_code,source_url,geometry,key,spatial_ref
0,E01030296,https://services1.arcgis.com/ESMARspQHYMw9BZ9/...,"MULTIPOLYGON (((517080.666 164767.597, 517080....",1030296,0
1,E01030296,https://services1.arcgis.com/ESMARspQHYMw9BZ9/...,"MULTIPOLYGON (((517080.666 164767.597, 517080....",1030296,0
2,E01030296,https://services1.arcgis.com/ESMARspQHYMw9BZ9/...,"MULTIPOLYGON (((517080.666 164767.597, 517080....",1030296,0
3,E01030296,https://services1.arcgis.com/ESMARspQHYMw9BZ9/...,"MULTIPOLYGON (((517080.666 164767.597, 517080....",1030296,0
4,E01030296,https://services1.arcgis.com/ESMARspQHYMw9BZ9/...,"MULTIPOLYGON (((517080.666 164767.597, 517080....",1030296,0
...,...,...,...,...,...
2076475,E01035364,https://services1.arcgis.com/ESMARspQHYMw9BZ9/...,"MULTIPOLYGON (((534472 158635.821, 534465.159 ...",1035364,0
2076476,E01035364,https://services1.arcgis.com/ESMARspQHYMw9BZ9/...,"MULTIPOLYGON (((534472 158635.821, 534465.159 ...",1035364,0
2076477,E01035364,https://services1.arcgis.com/ESMARspQHYMw9BZ9/...,"MULTIPOLYGON (((534472 158635.821, 534465.159 ...",1035364,0
2076478,E01035364,https://services1.arcgis.com/ESMARspQHYMw9BZ9/...,"MULTIPOLYGON (((534472 158635.821, 534465.159 ...",1035364,0


In [ ]:
# https://www.gov.uk/government/statistics/energy-chapter-1-digest-of-united-kingdom-energy-statistics-dukes
# the DUKES data looks at consumption data and would require aggregation
# I'm not sure if this is the best data for an overview

In [ ]:
# https://www.data.gov.uk/dataset/cfd94301-a2f2-48a2-9915-e477ca6d8b7e/pollution-inventory
# data from The Environment Agency's Pollution Inventory (PI) for England 
# It appears that this would identify major industries or companies where polution, waste or emissions are relevant

In [7]:
def clean(col_name: str):
    whitespace_cleaned = re.sub(r"\s+", "_", col_name.strip().lower())
    return re.sub(r"\W+", "", whitespace_cleaned)

In [115]:
imd = iod25.get_imd_domains()
imd.filter(domain="overall_index").select("lsoa21cd", pl.col("decile").cast(pl.Int32))

lsoa21cd,decile
str,i32
"""E01000001""",8
"""E01000002""",10
"""E01000003""",8
"""E01000005""",5
"""E01000006""",4
…,…
"""E01035758""",9
"""E01035759""",9
"""E01035760""",9


In [46]:
la_emissions = pl.read_csv("https://assets.publishing.service.gov.uk/media/68653c7ee6c3cc924228943f/2005-23-uk-local-authority-ghg-emissions-CSV-dataset.csv").rename(clean)
la_emissions

country,country_code,region,region_code,second_tier_authority,local_authority,local_authority_code,calendar_year,la_ghg_sector,la_ghg_subsector,greenhouse_gas,territorial_emissions_kt_co2e,emissions_within_the_scope_of_influence_of_las_kt_co2,midyear_population_thousands,area_km2
str,str,str,str,str,str,str,i64,str,str,str,f64,f64,f64,f64
"""England""","""E92000001""","""North East""","""E12000001""","""Hartlepool""","""Hartlepool""","""E06000001""",2005,"""Agriculture""","""Agriculture Electricity""","""CO2""",1.690511,1.690511,90.457,98.3466
"""England""","""E92000001""","""North East""","""E12000001""","""Hartlepool""","""Hartlepool""","""E06000001""",2005,"""Agriculture""","""Agriculture Electricity""","""CH4""",0.0535,0.0,90.457,98.3466
"""England""","""E92000001""","""North East""","""E12000001""","""Hartlepool""","""Hartlepool""","""E06000001""",2005,"""Agriculture""","""Agriculture Electricity""","""N2O""",0.00682,0.0,90.457,98.3466
"""England""","""E92000001""","""North East""","""E12000001""","""Hartlepool""","""Hartlepool""","""E06000001""",2005,"""Agriculture""","""Agriculture Gas""","""CO2""",0.264576,0.264576,90.457,98.3466
"""England""","""E92000001""","""North East""","""E12000001""","""Hartlepool""","""Hartlepool""","""E06000001""",2005,"""Agriculture""","""Agriculture Gas""","""CH4""",0.00836,0.0,90.457,98.3466
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Wales""","""W92000004""","""Wales""","""W92000004""","""Wales""","""Merthyr Tydfil""","""W06000024""",2023,"""Transport""","""Transport 'Other'""","""N2O""",0.00766,0.0,58.593,111.9569
"""Wales""","""W92000004""","""Wales""","""W92000004""","""Wales""","""Merthyr Tydfil""","""W06000024""",2023,"""Waste""","""Landfill""","""CH4""",6.56296,0.0,58.593,111.9569
"""Wales""","""W92000004""","""Wales""","""W92000004""","""Wales""","""Merthyr Tydfil""","""W06000024""",2023,"""Waste""","""Waste 'Other'""","""CO2""",0.0289,0.0289,58.593,111.9569


In [117]:
lsoa_consumption = pl.read_csv(
    "https://assets.publishing.service.gov.uk/media/694295f4fdbd8404f9e1f200/elec_domestic_LSOA_stacked_2010-2024.csv",
    encoding='utf8-lossy'
).rename(clean).filter(pl.col("la_code").is_in(SURREY_DISTRICTS)).join(
    imd.filter(domain="overall_index").select(lsoa_code="lsoa21cd", imd_decile=pl.col("decile").cast(pl.Int32)),
    on="lsoa_code"
)
lsoa_consumption

year,la_code,la,msoa_code,msoa,lsoa_code,lsoa,meters,consumption_kwh,mean_consumption_kwh_per_meter,median_consumption_kwh_per_meter,imd_decile
i64,str,str,str,str,str,str,i64,f64,f64,f64,i32
2010,"""E07000207""","""Elmbridge""","""E02006331""","""Elmbridge 015""","""E01030296""","""Elmbridge 015A""",668,4273425.5,6397.343563,5208.55,10
2011,"""E07000207""","""Elmbridge""","""E02006331""","""Elmbridge 015""","""E01030296""","""Elmbridge 015A""",673,4449644.5,6611.656018,5346.4,10
2012,"""E07000207""","""Elmbridge""","""E02006331""","""Elmbridge 015""","""E01030296""","""Elmbridge 015A""",674,4307807.6,6391.405935,5162.8,10
2013,"""E07000207""","""Elmbridge""","""E02006331""","""Elmbridge 015""","""E01030296""","""Elmbridge 015A""",669,4287333.3,6408.569955,5425.7,10
2014,"""E07000207""","""Elmbridge""","""E02006331""","""Elmbridge 015""","""E01030296""","""Elmbridge 015A""",677,4.260867e6,6293.747415,5329.0,10
…,…,…,…,…,…,…,…,…,…,…,…
2020,"""E07000215""","""Tandridge""","""E02006429""","""Tandridge 002""","""E01035364""","""Tandridge 002F""",600,2125726.8,3542.878,2781.4,7
2021,"""E07000215""","""Tandridge""","""E02006429""","""Tandridge 002""","""E01035364""","""Tandridge 002F""",598,2.0438e6,3417.733912,2608.75,7
2022,"""E07000215""","""Tandridge""","""E02006429""","""Tandridge 002""","""E01035364""","""Tandridge 002F""",601,1.8810e6,3129.864468,2412.9,7


In [124]:
px.line(
    lsoa_consumption,
    line_group="lsoa_code",
    x="year",
    y="mean_consumption_kwh_per_meter",
    color="imd_decile"
)

In [ ]:
# predictable pattern of more consumption less deprivation

px.line(
    lsoa_consumption.group_by("year", "imd_decile").agg(mean_consumption=pl.sum("consumption_kwh") / pl.sum("meters")).sort("imd_decile", "year"),
    x="year",
    y="mean_consumption",
    color="imd_decile"
)

In [ ]:
px.line(
    lsoa_consumption.group_by("year", "la").agg(mean_consumption=pl.sum("consumption_kwh") / pl.sum("meters")).sort("la", "year"),
    x="year",
    y="mean_consumption",
    color="la"
)

In [135]:
pre_post = (
    pl.col("mean_consumption_kwh_per_meter")
    .rolling("year", period="2i", offset="-2i")
    .over("lsoa_code", order_by="year")
)
px.line(
    lsoa_consumption.select(
        "year", "lsoa_code", "mean_consumption_kwh_per_meter",
        trend=pl.col("mean_consumption_kwh_per_meter") - pre_post.list.first(),
    ).sort("lsoa_code", "year"),
    x="year",
    y="trend"
)

In [137]:
district_consumption = pl.read_csv(
    "https://assets.publishing.service.gov.uk/media/694295bd9273c48f554cf4ef/elec_LA_stacked_2005-2024.csv",
    encoding='utf8-lossy'
)
district_consumption

Year,LA_Code,Region,LA,Standard_domestic_meters_thousands,E7_domestic_meters_thousands,Domestic_meters_thouasands,Non_domestic_meters_thousands,All_meters_thousands,Standard_domestic_consumption_GWh,E7_domestic_consumption_GWh,Domestic_consumption_GWh,Non_domestic_consumption_GWh,All_consumption_GWh,Standard_domestic_mean_consumption_KWh,E7_domestic_mean_consumption_KWh_per_meter,Domestic_mean_consumption_KWh_per_meter,Non_domestic_mean_consumption_KWh_per_meter,All_mean_consumption_kWh_per_meter,Standard_domestic_median_consumption_kWh_per_meter,E7_domestic_median_consumption_kWh_per_meter,Domestic_median_consumption_kWh_per_meter,Non_domestic_median_consumption_kWh_per_meter,All_median_consumption_kWh_per_meter,Domestic_mean_consumption_kWh_per_household
i64,str,str,str,str,str,f64,f64,f64,str,str,f64,f64,f64,str,str,f64,f64,f64,str,str,str,str,str,str
2005,"""UKC2101""","""North East""","""Alnwick""",null,null,16.493,1.923,18.416,null,null,77.868961,86.752881,164.621842,null,null,4721.333966,45113.3025,8939.066127,null,null,null,null,null,null
2005,"""UKC2102""","""North East""","""Berwick-upon-Tweed""",null,null,15.848,2.403,18.251,null,null,83.214705,100.657491,183.872196,null,null,5250.801685,41888.26088,10074.63679,null,null,null,null,null,null
2005,"""UKC2103""","""North East""","""Blyth Valley""",null,null,36.5,1.928,38.428,null,null,137.895655,199.234089,337.129745,null,null,3777.963159,103337.1832,8773.023436,null,null,null,null,null,null
2005,"""UKC2104""","""North East""","""Castle Morpeth""",null,null,22.089,1.952,24.041,null,null,106.369599,131.033808,237.403407,null,null,4815.500874,67127.97546,9874.938933,null,null,null,null,null,null
2005,"""UKC1409""","""North East""","""Chester-le-Street""",null,null,24.483,1.26,25.743,null,null,91.517394,74.992367,166.509761,null,null,3737.997553,59517.75119,6468.156804,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2024,"""S12000045""","""Scotland""","""East Dunbartonshire""","""45.6""","""2.744""",48.344,2.5,50.844,"""140.1613752""","""12.47090644""",152.632282,136.051666,288.683947,"""3073.714368""","""4544.790975""",3157.21251,54420.66629,5677.837057,"""2496.75""","""3276.5""","""2519.75""","""7397.35""","""2570.380934""","""3265.364082"""
2024,"""S12000047""","""Scotland""","""Fife""","""173.494""","""11.507""",185.001,12.391,197.392,"""483.7924342""","""50.80629038""",534.598725,813.636737,1348.235462,"""2788.525449""","""4415.25075""",2889.707215,65663.52492,6830.243687,"""2221.45""","""2919.6""","""2245.1""","""7517.6""","""2302.4""","""3109.697697"""
2024,"""S12000048""","""Scotland""","""Perth and Kinross""","""69.185""","""12.192""",81.377,8.585,89.962,"""237.2733471""","""73.5117458""",310.785093,358.79046,669.575553,"""3429.548993""","""6029.506709""",3819.077785,41792.71517,7442.870908,"""2504.8""","""4862.05""","""2659.4""","""7448.2""","""2764.1""","""4350.214888"""


In [159]:
england_total = district_consumption.filter(~pl.col("Region").is_in(["Wales", "Scotland"])).group_by("Year").agg(
    All_mean_consumption_kWh_per_meter=(pl.sum("All_consumption_GWh") / (pl.sum("All_meters_thousands"))) * 1000,
    Non_domestic_mean_consumption_KWh_per_meter=(pl.sum("Non_domestic_consumption_GWh") / (pl.sum("Non_domestic_meters_thousands"))) * 1000,
    Domestic_mean_consumption_KWh_per_meter=(pl.sum("Domestic_consumption_GWh") / (pl.sum("Domestic_meters_thouasands"))) * 1000,
)
england_total

Year,All_mean_consumption_kWh_per_meter,Non_domestic_mean_consumption_KWh_per_meter,Domestic_mean_consumption_KWh_per_meter
i64,f64,f64,f64
2007,10584.345213,78068.325679,4404.402551
2013,9606.318484,74037.127499,3962.635593
2010,9992.825787,76392.694335,4163.446871
2022,7878.315204,62508.583955,3268.122757
2016,9170.556508,71200.704929,3862.796445
…,…,…,…
2021,8211.16063,62853.034947,3581.651923
2009,9927.246254,74933.540993,4162.644958
2012,9661.216951,73832.264694,4034.900695


In [171]:
comp_df = pl.concat(
    [district_consumption.filter(
    pl.col("LA_Code").is_in(SURREY_DISTRICTS)
).select(
    "Year", "All_mean_consumption_kWh_per_meter", "Non_domestic_mean_consumption_KWh_per_meter", "Domestic_mean_consumption_KWh_per_meter","LA"
), england_total.with_columns(LA=pl.lit("england"))], how="vertical_relaxed").sort("LA", "Year")
comp_df

Year,All_mean_consumption_kWh_per_meter,Non_domestic_mean_consumption_KWh_per_meter,Domestic_mean_consumption_KWh_per_meter,LA
i64,f64,f64,f64,str
2012,9465.0,55115.0,5287.0,"""Elmbridge"""
2013,9338.0,54740.0,5194.0,"""Elmbridge"""
2014,9340.554242,53646.07763,5191.478607,"""Elmbridge"""
2015,9231.399517,54255.36677,5158.621413,"""Elmbridge"""
2016,8878.353142,50797.84321,5082.124122,"""Elmbridge"""
…,…,…,…,…
2020,8252.320221,60152.156017,3829.366248,"""england"""
2021,8211.16063,62853.034947,3581.651923,"""england"""
2022,7878.315204,62508.583955,3268.122757,"""england"""


In [172]:
px.line(
    comp_df,
    x="Year",
    y="All_mean_consumption_kWh_per_meter",
    color="LA"
)

In [174]:
px.line(
    comp_df,
    x="Year",
    y="Domestic_mean_consumption_KWh_per_meter",
    color="LA"
)

In [175]:
px.line(
    comp_df,
    x="Year",
    y="Non_domestic_mean_consumption_KWh_per_meter",
    color="LA"
)

In [ ]:
from jsna_climate_and_environment.geography.places import SURREY_DISTRICTS


unique_key = ("country", "country_code", "calendar_year", "la_ghg_sector", "la_ghg_subsector", "greenhouse_gas")
df.with_columns(
    country_total=pl.sum("territorial_emissions_kt_co2e").over([*unique_key,]),
    country_pop=pl.sum("midyear_population_thousands").over([*unique_key]),
    region_total=pl.sum("territorial_emissions_kt_co2e").over([*unique_key, "region", "region_code"]),
    region_pop=pl.sum("midyear_population_thousands").over([*unique_key, "region", "region_code"])
).filter(
    pl.col("local_authority_code").is_in(SURREY_DISTRICTS)
)

country,country_code,region,region_code,second_tier_authority,local_authority,local_authority_code,calendar_year,la_ghg_sector,la_ghg_subsector,greenhouse_gas,territorial_emissions_kt_co2e,emissions_within_the_scope_of_influence_of_las_kt_co2,midyear_population_thousands,area_km2,country_total,country_pop,region_total,region_pop
str,str,str,str,str,str,str,i64,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""England""","""E92000001""","""South East""","""E12000008""","""Surrey""","""Elmbridge""","""E07000207""",2005,"""Agriculture""","""Agriculture Electricity""","""CO2""",1.292893,1.292893,127.286,96.3343,2060.531584,50606.034,276.728398,8202.896
"""England""","""E92000001""","""South East""","""E12000008""","""Surrey""","""Elmbridge""","""E07000207""",2005,"""Agriculture""","""Agriculture Electricity""","""CH4""",0.0409,0.0,127.286,96.3343,65.208599,50606.034,8.757279,8202.896
"""England""","""E92000001""","""South East""","""E12000008""","""Surrey""","""Elmbridge""","""E07000207""",2005,"""Agriculture""","""Agriculture Electricity""","""N2O""",0.00522,0.0,127.286,96.3343,8.313532,50606.034,1.116726,8202.896
"""England""","""E92000001""","""South East""","""E12000008""","""Surrey""","""Elmbridge""","""E07000207""",2005,"""Agriculture""","""Agriculture Gas""","""CO2""",1.084353,1.084353,127.286,96.3343,664.107279,49072.869,166.264032,7610.706
"""England""","""E92000001""","""South East""","""E12000008""","""Surrey""","""Elmbridge""","""E07000207""",2005,"""Agriculture""","""Agriculture Gas""","""CH4""",0.0342,0.0,127.286,96.3343,20.973769,49072.869,5.250769,7610.706
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""England""","""E92000001""","""South East""","""E12000008""","""Surrey""","""Woking""","""E07000217""",2023,"""Transport""","""Transport 'Other'""","""N2O""",0.0141,0.0,104.636,63.6034,26.751566,57690.323,4.861903,9482.507
"""England""","""E92000001""","""South East""","""E12000008""","""Surrey""","""Woking""","""E07000217""",2023,"""Waste""","""Landfill""","""CH4""",6.6765428,0.0,104.636,63.6034,12021.390194,57690.323,1653.192218,9482.507
"""England""","""E92000001""","""South East""","""E12000008""","""Surrey""","""Woking""","""E07000217""",2023,"""Waste""","""Waste 'Other'""","""CO2""",0.0439,0.0439,104.636,63.6034,222.29429,57690.323,74.949359,9482.507


In [45]:
df.group_by(
    "la_ghg_subsector",
).agg(
    pl.col("la_ghg_sector").unique().item(),
    pl.col("calendar_year").n_unique(),
    pl.col("greenhouse_gas").n_unique()
).filter(
    (pl.col("greenhouse_gas") != 3) | (pl.col("calendar_year") != 19) 
)

la_ghg_subsector,la_ghg_sector,calendar_year,greenhouse_gas
str,str,u32,u32
"""LULUCF Net Emissions: Grasslan…","""LULUCF""",19,2
"""LULUCF Net Emissions: Cropland…","""LULUCF""",19,2
"""Agriculture Livestock""","""Agriculture""",19,2
"""LULUCF Net Emissions: Settleme…","""LULUCF""",19,2
"""Agriculture Soils""","""Agriculture""",19,2
"""LULUCF Net Emissions: Bioenerg…","""LULUCF""",17,1
"""Landfill""","""Waste""",19,1
